# 第 08 课｜协方差与相关性

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`08_youtube-8lopnSDzetE/index.html`  
> **所属模块**：P1-M2 · 收益、时间价值与风险

## 本节量化问题

组合分散化依赖资产数量，还是依赖收益共同变化结构？

## 准确定义与策略报告位置

协方差保留量纲，相关系数标准化到 [-1,1]；组合方差由权重和协方差矩阵共同决定。

- **核心关系**：Cov(X,Y)=E[(X-μx)(Y-μy)]；ρ=Cov(X,Y)/(σxσy)；组合方差 w'Σw。
- **写入报告**：组合依赖结构和分散化证据
- **P2 衔接**：P2 第 33、37 课与 L1：矩阵、分组和组合风险。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 协方差、相关性和组合方差

先用 Pandas 按日期与资产标签得到收益矩阵，再进入矩阵运算。`weights @ covariance @ weights` 是二次型。

In [2]:
daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
close = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
returns = close.pct_change(fill_method=None).dropna()

covariance = returns.cov()
pearson = returns.corr(method="pearson")
spearman = returns.corr(method="spearman")
weights = pd.Series(1.0 / returns.shape[1], index=returns.columns)
portfolio_variance = float(weights @ covariance @ weights)

print(f"等权组合日方差: {portfolio_variance:.8f}")
display(pearson.round(3))
display((spearman - pearson).round(3).head())

等权组合日方差: 0.00003599


symbol,SIM_BOND,SIM_COMMODITY,SIM_DEFENSIVE,SIM_GROWTH,SIM_MARKET,SIM_VALUE
symbol,,,,,,
SIM_BOND,1.000,-0.132,-0.211,-0.238,-0.304,-0.218
SIM_COMMODITY,-0.132,1.000,0.222,0.271,0.301,0.268
SIM_DEFENSIVE,-0.211,0.222,1.000,0.608,0.742,0.559
SIM_GROWTH,-0.238,0.271,0.608,1.000,0.751,0.549
SIM_MARKET,-0.304,0.301,0.742,0.751,1.000,0.732
SIM_VALUE,-0.218,0.268,0.559,0.549,0.732,1.000


symbol,SIM_BOND,SIM_COMMODITY,SIM_DEFENSIVE,SIM_GROWTH,SIM_MARKET,SIM_VALUE
symbol,,,,,,
SIM_BOND,0.000,0.006,0.018,0.026,0.030,0.022
SIM_COMMODITY,0.006,0.000,-0.012,-0.021,-0.024,-0.021
SIM_DEFENSIVE,0.018,-0.012,0.000,-0.006,-0.019,-0.021
SIM_GROWTH,0.026,-0.021,-0.006,0.000,-0.028,-0.045
SIM_MARKET,0.030,-0.024,-0.019,-0.028,0.000,-0.024


## 方法边界与阶段验收

相关不等于因果，且会随窗口和市场状态变化；危机期相关性可能显著上升。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
